In [ ]:
# Unzip dataset if needed
!unzip -o /content/*ai*.zip -d /content/

In [ ]:
# Install dependencies
!pip install -U transformers accelerate peft

In [ ]:
import os, re, math, random
from contextlib import nullcontext

import pandas as pd
from PIL import Image, ImageEnhance
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "/content"
CHOICES = ("a", "b", "c", "d")

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
def enhance_image(image):
    image = ImageEnhance.Contrast(image).enhance(1.3)
    image = ImageEnhance.Sharpness(image).enhance(1.5)
    return image

def resolve_path(path):
    return path if os.path.isabs(path) else os.path.join(DATA_ROOT, path)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()

def classify_question_type(question):
    q = normalize_text(question)
    if any(token in q for token in ["??? ??", "?? ?", "?? ?", "???? ??"]):
        return "negative"
    if any(token in q for token in ["??", "??", "??"]):
        return "location"
    if "?" in q or "??" in q:
        return "color"
    if any(token in q for token in ["? ?", "??", "? ?", "? ?", "? ??"]):
        return "count"
    if any(token in q for token in ["??", "??"]):
        return "material"
    if any(token in q for token in ["????", "??? ??", "??"]):
        return "recycling"
    if any(token in q for token in ["??", "??"]):
        return "what"
    return "other"

def extract_gold_text(row):
    answer_key = normalize_text(row["answer"])
    return str(row.get(answer_key, "")).strip()

def build_analysis_frame(df):
    analyzed = df.copy().reset_index(drop=True)
    analyzed["answer"] = analyzed["answer"].astype(str).str.strip().str.lower()
    analyzed["gold_text"] = analyzed.apply(extract_gold_text, axis=1)
    analyzed["question_type"] = analyzed["question"].map(classify_question_type)
    return analyzed

def stratified_split_dataframe(df, stratify_col, valid_ratio=0.1, seed=42):
    rng = random.Random(seed)
    train_indices, valid_indices = [], []
    for _, group in df.groupby(stratify_col):
        indices = group.index.tolist()
        rng.shuffle(indices)
        valid_size = max(1, int(round(len(indices) * valid_ratio))) if len(indices) > 1 else 0
        valid_indices.extend(indices[:valid_size])
        train_indices.extend(indices[valid_size:])
    train_df = df.loc[sorted(train_indices)].reset_index(drop=True)
    valid_df = df.loc[sorted(valid_indices)].reset_index(drop=True)
    return train_df, valid_df

def build_weighted_sampler(train_df):
    question_counts = train_df["question_type"].value_counts().to_dict()
    gold_counts = train_df["gold_text"].value_counts().to_dict()
    weights = []
    for _, row in train_df.iterrows():
        type_weight = 1.0 / max(question_counts.get(row["question_type"], 1), 1)
        gold_weight = 1.0 / max(gold_counts.get(row["gold_text"], 1), 1)
        weights.append(math.sqrt(type_weight) * math.sqrt(gold_weight))
    weights = torch.tensor(weights, dtype=torch.double)
    if len(weights) > 0:
        weights = torch.clamp(weights, max=torch.quantile(weights, 0.95))
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

def summarize_slices(df, pred_col="pred_answer"):
    summary = []
    for question_type, group in df.groupby("question_type"):
        acc = (group[pred_col] == group["answer"]).mean()
        summary.append((question_type, len(group), float(acc)))
    summary.sort(key=lambda x: x[0])
    return summary

def extract_choice(text):
    normalized = normalize_text(text)
    if normalized in CHOICES:
        return normalized
    matches = re.findall(r"\b([abcd])\b", normalized)
    if matches:
        return matches[-1]
    return "a"

def maybe_shuffle_options(a, b, c, d, gold, enabled=False):
    option_items = list(zip(CHOICES, [a, b, c, d]))
    if not enabled:
        return {choice: text for choice, text in option_items}, gold
    random.shuffle(option_items)
    shuffled = {}
    remapped_gold = gold
    for new_choice, (old_choice, option_text) in zip(CHOICES, option_items):
        shuffled[new_choice] = option_text
        if old_choice == gold:
            remapped_gold = new_choice
    return shuffled, remapped_gold

def autocast_context():
    if device == "cuda":
        return torch.amp.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(degrees=45),
])

SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant specializing in recycling and waste classification. "
    "Carefully observe the material, shape, and condition of the object in the image. "
    "Answer using exactly one lowercase letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Return only one lowercase letter among a, b, c, or d."
    )

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True, option_shuffle_train=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.option_shuffle_train = option_shuffle_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(resolve_path(row["path"])).convert("RGB")
        img = enhance_image(img)
        if self.train:
            img = train_transform(img)

        q = str(row["question"])
        gold = str(row["answer"]).strip().lower() if self.train else None
        option_map, gold = maybe_shuffle_options(
            str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"]),
            gold,
            enabled=self.train and self.option_shuffle_train,
        )
        user_text = build_mc_prompt(q, option_map["a"], option_map["b"], option_map["c"], option_map["d"])

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]},
        ]
        if self.train:
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

        return {
            "messages": messages,
            "image": img,
            "gold": gold,
            "question_type": row.get("question_type"),
            "gold_text": row.get("gold_text"),
        }

class DataCollator:
    def __init__(self, processor, train=True):
        self.processor = processor
        self.train = train

    def __call__(self, samples):
        texts, prompt_texts, images = [], [], []
        for sample in samples:
            prompt_text = self.processor.apply_chat_template(
                sample["messages"], tokenize=False, add_generation_prompt=not self.train
            )
            prompt_texts.append(prompt_text)
            images.append(sample["image"])
            if self.train:
                texts.append(prompt_text + sample["gold"])
            else:
                texts.append(prompt_text)

        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            prompt_encoded = self.processor(text=prompt_texts, images=images, padding=True, return_tensors="pt")
            labels = encoded["input_ids"].clone()
            labels[encoded["attention_mask"] == 0] = -100

            prompt_lengths = prompt_encoded["attention_mask"].sum(dim=1).tolist()
            full_lengths = encoded["attention_mask"].sum(dim=1).tolist()
            sequence_length = labels.shape[1]
            for idx, (prompt_length, full_length) in enumerate(zip(prompt_lengths, full_lengths)):
                answer_length = max(int(full_length - prompt_length), 0)
                answer_start = sequence_length - answer_length
                labels[idx, :answer_start] = -100
            encoded["labels"] = labels

        return encoded

def predict_choice(model, processor, row):
    img = Image.open(resolve_path(row["path"])).convert("RGB")
    img = enhance_image(img)
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)
    with torch.no_grad(), autocast_context():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    output_text = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    return extract_choice(output_text)

def evaluate_on_dataframe(model, processor, df, desc="Validation"):
    result_df = df.copy().reset_index(drop=True)
    preds = []
    for _, row in tqdm(result_df.iterrows(), total=len(result_df), desc=desc, unit="sample"):
        preds.append(predict_choice(model, processor, row))
    result_df["pred_answer"] = preds
    result_df["correct"] = result_df["pred_answer"] == result_df["answer"]
    accuracy = float(result_df["correct"].mean()) if len(result_df) else 0.0
    return accuracy, result_df

In [ ]:
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")
train_df = build_analysis_frame(train_df)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    max_pixels=768 * 28 * 28,
)
processor.tokenizer.padding_side = "left"

train_subset, valid_subset = stratified_split_dataframe(
    train_df,
    stratify_col="question_type",
    valid_ratio=0.1,
    seed=SEED,
)
train_sampler = build_weighted_sampler(train_subset)

BATCH_SIZE = 1
GRAD_ACCUM = 16

train_loader = DataLoader(
    VQAMCDataset(train_subset, processor, train=True, option_shuffle_train=True),
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    shuffle=False,
    collate_fn=DataCollator(processor, True),
)
valid_loss_loader = DataLoader(
    VQAMCDataset(valid_subset, processor, train=True, option_shuffle_train=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=DataCollator(processor, True),
)

print(f"Train rows: {len(train_subset)} | Valid rows: {len(valid_subset)}")
print("\n[Question type distribution]")
print(train_df["question_type"].value_counts().to_string())
print("\n[Top 15 gold texts]")
print(train_df["gold_text"].value_counts().head(15).to_string())
print("\n[15 rarest gold texts]")
print(train_df["gold_text"].value_counts().sort_values().head(15).to_string())

In [ ]:
print("Loading model...")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
EPOCHS = 3
SAVE_DIR = "/content/qwen2_5_vl_7b_lora_balanced"

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.1),
    num_training_steps=num_training_steps,
)

best_valid_acc = -1.0
best_valid_df = None

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1} [train]", unit="batch")
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with autocast_context():
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        avg_loss = running / min(step, GRAD_ACCUM) if step < GRAD_ACCUM else running / GRAD_ACCUM
        progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
        if step % GRAD_ACCUM == 0:
            running = 0.0

    model.eval()
    val_loss = 0.0
    with torch.no_grad(), autocast_context():
        for vb in tqdm(valid_loss_loader, desc=f"Epoch {epoch + 1} [valid loss]", unit="batch"):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
    avg_val_loss = val_loss / max(len(valid_loss_loader), 1)

    valid_acc, valid_pred_df = evaluate_on_dataframe(
        model,
        processor,
        valid_subset,
        desc=f"Epoch {epoch + 1} [valid infer]",
    )
    print(f"[Epoch {epoch + 1}] valid loss {avg_val_loss:.4f} | valid acc {valid_acc:.4f}")
    print("Question type accuracy:")
    for question_type, count, acc in summarize_slices(valid_pred_df):
        print(f"- {question_type}: {acc:.4f} ({count} samples)")

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_valid_df = valid_pred_df.copy()
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"Best model updated at epoch {epoch + 1}: {SAVE_DIR}")

if best_valid_df is not None:
    best_valid_df.to_csv("/content/valid_predictions_balanced.csv", index=False)
    print(f"Best valid accuracy: {best_valid_acc:.4f}")
    print("Saved validation predictions to: /content/valid_predictions_balanced.csv")

In [ ]:
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(resolve_path(row["path"])).convert("RGB")
    img = enhance_image(img)

    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad(), autocast_context():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved prediction to: /content/submission.csv")